# BioMapper REST API tutorial

This notebook calls the hosted BioMapper API (the `biomapper2` engine) **directly over HTTP**, with nothing but `requests` and `pandas`. It is for anyone who wants to use the service from a language or pipeline that is not Python, or who wants to see exactly what goes over the wire.

If you are working in Python and would rather not handle HTTP yourself, use the [`biomapper` client package](https://github.com/trentleslie/biomapper) (`pip install biomapper`) and its [tutorial notebook](https://github.com/trentleslie/biomapper/blob/main/notebooks/biomapper_tutorial.ipynb). It wraps these same endpoints.

What this covers:

1. A health gate and the provenance pins you should record with every result
2. Discovery: entity types, vocabularies, annotators
3. Resolving one entity, and how to read the response (including the resolution certificate)
4. Batching safely: small batches, a disk cache, retries, and a health check between batches
5. Resolving across entity types: metabolite, protein, gene, clinical lab
6. Requesting a target namespace, and a known limitation for genes (forced hits)
7. Writing a provenance record for your run

**How to run it.** From a clone of `biomapper2`: `uv sync --dev`, then `uv run --with jupyter jupyter lab examples/biomapper_api_tutorial.ipynb`. The notebook does not import `biomapper2`; it only needs `requests` and `pandas`, which that environment provides.

In [1]:
import hashlib
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# The hosted API. Override with BIOMAPPER_API_URL to point at your own deployment.
BASE_URL = os.environ.get("BIOMAPPER_API_URL", "https://biomapper.expertintheloop.io/api/v1").rstrip("/")

# The KRAKEN knowledge-graph service the hosted API resolves against. Its /health endpoint is where
# the graph build is reported (the BioMapper API's own /health does not carry build pins).
KESTREL_URL = os.environ.get("KESTREL_API_URL", "https://kestrel.krakenkg.com/api").rstrip("/")

# Optional. The hosted API is keyless; a self-hosted deployment may require a key (see below).
API_KEY = os.environ.get("BIOMAPPER_API_KEY")

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})
if API_KEY:
    session.headers["X-API-Key"] = API_KEY

print("python:", str(sys.executable).replace(str(Path.home()), "~"))
print("API:", BASE_URL)
print("Kestrel:", KESTREL_URL)
print("API key sent:", bool(API_KEY))

python: ~/.cache/uv/archive-v0/NuXA0JYPcloRdGJvub-Ua/bin/python
API: https://biomapper.expertintheloop.io/api/v1
Kestrel: https://kestrel.krakenkg.com/api
API key sent: False


### Authentication

The hosted API at `https://biomapper.expertintheloop.io/api/v1` runs in **open-access mode**: no key is needed for any endpoint in this notebook. `GET /health` never requires a key.

A deployment can require one. If the server sets `BIOMAPPER2_API_KEYS` (or `BIOMAPPER_API_KEY`), every discovery and mapping call must carry a valid `X-API-Key` header: a missing key returns **401** and a wrong key returns **403**. Keys are issued by whoever operates that deployment; there is no self-service signup. To use one here, export it before starting Jupyter (`export BIOMAPPER_API_KEY=...`). The cell above reads it from the environment and never writes it anywhere, including the provenance record at the end. Do not paste a key into a notebook cell.

## 1. Health gate and provenance pins

Two services are involved: the BioMapper API, and the Kestrel service that fronts the KRAKEN knowledge graph. Resolution results depend on the graph build, so record the build with every result. Read it from Kestrel's `/health`, which is keyless.

`TUTORIAL_EXECUTED_AGAINST` is what this notebook's saved outputs were produced against. It is a comparison, not a requirement: if your pins differ, the graph has moved on and some specific node choices below may differ too.

In [2]:
TUTORIAL_EXECUTED_AGAINST = {
    "kestrel_version": "0.3.0",
    "kg_version": "2.3.0",
    "kraken_package_version": "2.2.0",
    "biolink_version": "4.2.5",
    "git_commit": "3dd08a5b7b17e320ceb0c0091e09b781d4d90242",
}


def api_health(timeout=30):
    # True only if the API answers AND reports the mapper as ready.
    try:
        r = session.get(f"{BASE_URL}/health", timeout=timeout)
        body = r.json()
        return r.ok and body.get("status") == "healthy" and body.get("mapper_initialized") is True, body
    except (requests.RequestException, ValueError) as exc:
        return False, {"error": repr(exc)}


def kg_pins(timeout=30):
    r = requests.get(f"{KESTREL_URL}/health", timeout=timeout)
    r.raise_for_status()
    body = r.json()
    build = body.get("kg_build") or {}
    return {
        "kestrel_version": body.get("kestrel_version"),
        "kg_version": build.get("kg_version"),
        "kraken_package_version": build.get("kraken_package_version"),
        "biolink_version": build.get("biolink_version"),
        "git_commit": build.get("git_commit"),
        "build_timestamp": build.get("build_timestamp"),
        "sources": build.get("sources"),
    }


ok, api_body = api_health()
print("API healthy:", ok, api_body)
assert ok, "The API is not healthy; do not start a run against it."

PINS = kg_pins()
for key, expected in TUTORIAL_EXECUTED_AGAINST.items():
    flag = "" if PINS[key] == expected else "   <- differs from this notebook's saved outputs"
    print(f"{key:24s} {PINS[key]}{flag}")
print("sources:", ", ".join(PINS["sources"] or []))

API healthy: True {'status': 'healthy', 'version': '0.1.0', 'mapper_initialized': True}


kestrel_version          0.3.0
kg_version               2.3.0
kraken_package_version   2.2.0
biolink_version          4.2.5
git_commit               3dd08a5b7b17e320ceb0c0091e09b781d4d90242
sources: babel, bio-age, bio-bmi, cdes, ctkg, dakg, kg2, lipidmaps, loinc, long-covid, microbiome-kg, multiomics-kg, ncbigene, pgs-catalog, refmet, robokop, translator-kg-open, umls


Note that `kg_version` and `kraken_package_version` are different numbers. Record both. The `sources` list matters too: when a benchmark's gold standard comes from a source the graph ingests, agreement with that gold measures **coverage**, not independent accuracy.

## 2. Discovery

`GET /entity-types` lists the Biolink categories. The `aliases` field is what you can pass as `entity_type` (`metabolite`, `protein`, `gene`, `lab`, ...); the full Biolink name also works.

In [3]:
types = session.get(f"{BASE_URL}/entity-types", timeout=60).json()
aliased = pd.DataFrame(
    [
        {
            "entity_type": ", ".join(t["aliases"]),
            "biolink": t["type"],
            "default prefixes": ", ".join((t["defaultPrefixes"] or [])[:6]) + ", ...",
        }
        for t in types
        if t.get("aliases")
    ]
)
print(len(types), "Biolink categories;", len(aliased), "with short aliases:")
aliased

86 Biolink categories; 9 with short aliases:


,entity_type,biolink,default prefixes
0,"clinicallab, lab",biolink:ClinicalFinding,"LOINC, UMLS, SNOMEDCT, MEDDRA, NCIT, ..."
1,disease,biolink:Disease,"UMLS, MONDO, SNOMEDCT, medgen, MEDDRA, DOID, ..."
2,drug,biolink:Drug,"UMLS, RXNORM, VANDF, RXCUI, PUBCHEM.COMPOUND, ..."
3,gene,biolink:Gene,"NCBIGene, ENSEMBL, UniProtKB, VGNC, UMLS, HGNC..."
4,"microbiome, taxonomy",biolink:OrganismTaxon,"NCBITaxon, UMLS, MESH, NCIT, CHEMBL.TARGET, LO..."
5,pathway,biolink:Pathway,"SMPDB, PathWhiz, REACT, GO, UMLS, KEGG, ..."
6,phenotype,biolink:PhenotypicFeature,"UMLS, SNOMEDCT, MEDDRA, NCIT, LOINC, CHV, ..."
7,protein,biolink:Protein,"UMLS, UniProtKB, NCBIGene, ENSEMBL, LOINC, MES..."
8,"lipid, metabolite",biolink:SmallMolecule,"PUBCHEM.COMPOUND, INCHIKEY, CAS, CHEBI, CHEMBL..."


In [4]:
vocabs = session.get(f"{BASE_URL}/vocabularies", timeout=60).json()
annotators = session.get(f"{BASE_URL}/annotators", timeout=60).json()
print("vocabularies:", vocabs["count"])
print("annotators:", json.dumps(annotators, indent=1)[:1200])

vocabularies: 317
annotators: {
 "annotators": [
  {
   "slug": "kestrel-hybrid-search",
   "name": "KestrelHybridSearchAnnotator",
   "description": null
  },
  {
   "slug": "kestrel-text-search",
   "name": "KestrelTextSearchAnnotator",
   "description": null
  },
  {
   "slug": "kestrel-vector-search",
   "name": "KestrelVectorSearchAnnotator",
   "description": null
  },
  {
   "slug": "metabolomics-workbench",
   "name": "MetabolomicsWorkbenchAnnotator",
   "description": "Annotator that queries the Metabolomics Workbench RefMet match API.\n\n    Retrieves RefMet IDs for metabolite entities using fuzzy name matching.\n    Returns raw API field names; the Normalizer handles mapping and ID cleaning.\n\n    The /match endpoint handles non-standard names (e.g., \"cholate\" -> \"Cholic acid\").\n    Only refmet_id is returned since KRAKEN has all RefMet equivalencies.\n\n    API Endpoint: GET https://www.metabolomicsworkbench.org/rest/refmet/match/{metabolite_name}\n    "
  },
  {
   "

## 3. Resolve one entity

`POST /map/entity` takes a name, an entity type, optional identifiers you already have, and options. Here it is for a metabolite given by name only.

In [5]:
def post(path, body, timeout=300):
    r = session.post(f"{BASE_URL}{path}", json=body, timeout=timeout)
    r.raise_for_status()
    return r.json()


t0 = time.time()
resp = post("/map/entity", {"name": "glucose", "entity_type": "metabolite"})
print(f"{time.time() - t0:.1f} s, request_id {resp['metadata']['request_id']}")
res = resp["result"]
print("chosen_kg_id:", res["chosen_kg_id"])
print("curies:", res["curies"])
print("assigned_ids:", json.dumps(res["assigned_ids"])[:300])
print("kg_equivalent_ids:", {ns: ids[:3] for ns, ids in res["kg_equivalent_ids"].items()})

0.0 s, request_id ada07d21-7f0b-45c8-a6b6-574fb527a666
chosen_kg_id: RM:0135901
curies: ['RM:0135901', 'HMDB:HMDB0000122']
assigned_ids: {"metabolomics-workbench": {"refmet_id": {"RM0135901": {}}}, "kestrel-hybrid-search": {"HMDB": {"HMDB0000122": {"score": 5.118003677788181, "resolved_via": "canonical_preference"}}}}
kg_equivalent_ids: {'ATC': ['B05CX01', 'V04CA02', 'V06DC01'], 'CAS': ['200-075-1', '218-914-5', '2280-44-6'], 'CHEBI': ['4167'], 'CHEMBL.COMPOUND': ['CHEMBL1222250'], 'CHV': ['0000005563', '0000028232', '0000028240'], 'FMA': ['82743'], 'FOODON': ['03315177'], 'HMDB': ['HMDB0304632'], 'INCHIKEY': ['WQZGKKKJIJFFOK-GASJEMHNSA-N'], 'KEGG': ['C00031'], 'KEGG.COMPOUND': ['C00031'], 'KEGG.DRUG': ['D00009'], 'KEGG.ENZYME': ['1.1.3.4'], 'NCIT': ['C2831', 'C61650', 'C61716'], 'NDDF': ['000915', '002597', '012094'], 'PSY': ['21160'], 'PUBCHEM.COMPOUND': ['5793'], 'PathWhiz.Compound': ['395', '57903', '77'], 'RM': ['0135901']}


### Reading the response

| Field | What it is |
|---|---|
| `chosen_kg_id` | The graph node the resolver picked. This is the answer. |
| `curies` | The normalized identifiers that fed resolution (from your `identifiers` and from annotation). |
| `assigned_ids` | Identifiers the annotators found for the name, before normalization. |
| `kg_equivalent_ids` | Cross-references the graph asserts for `chosen_kg_id`, as `{namespace: [local ids]}`. Read your target namespace's identifiers from here. |
| `kg_ids` | Graph nodes each CURIE linked to, before the resolver voted. |
| `resolution_certificate` | What the graph says about the chosen node's structure, and whether an independent registry agrees (small molecules only; see below). |
| `refmet_availability`, `refmet_source` | Whether RefMet answered for this row and which source served it. A runtime signal, not a verdict. |
| `error` | Set if mapping failed for this entity. A batch can succeed overall with some rows carrying an error. |

`metadata.request_id` identifies the call; keep it with your results.

### The resolution certificate

The certificate describes `chosen_kg_id` only. Its `state` is one of:

| `state` | Meaning |
|---|---|
| `corroborated` | An independent registry returned a structure whose InChIKey first block matches one the graph asserts for the chosen node. |
| `uncorroborated` | The graph asserts a structure, but no independent evidence confirmed it. Not a failure. |
| `contradicted` | An independent registry returned a **different** structure for the query name. A human should look. It does not prove the resolver wrong: registry name lookup can itself return a related compound. |
| `unavailable` | No structure was available to check against. Unverifiable, not wrong. |
| `not_applicable` | Outside the small-molecule population the certificate is defined for, e.g. genes, proteins, clinical labs. |

The comparison uses the InChIKey **first block** (connectivity), which is not tautomer- or charge-invariant, so treat `contradicted` as a prompt to review, not a verdict.

In [6]:
cert = res["resolution_certificate"]
print(
    {
        k: cert[k]
        for k in (
            "state",
            "structure_status",
            "node_inchikey_blocks",
            "independent_source",
            "independent_inchikey_block",
            "comparison_rule",
        )
    }
)

{'state': 'corroborated', 'structure_status': 'structure_present', 'node_inchikey_blocks': ['WQZGKKKJIJFFOK'], 'independent_source': 'metabolomics-workbench', 'independent_inchikey_block': 'WQZGKKKJIJFFOK', 'comparison_rule': 'inchikey_first_block_set_intersection/v1'}


## 4. Batching safely

`POST /map/batch` accepts up to 1,000 entities, but do not send that many at once. Measured on the production service: roughly **4.4 s per entity** under sustained load, growing faster than linearly with batch size, and the service can **wedge under heavy load**. The service also caches Kestrel lookups for about an hour, so a repeated or popular name can come back in a fraction of a second (the glucose call above may have). Plan for the slow case.

So the helper below:

- sends **batches of 10**;
- **caches** every batch response on disk, keyed by the exact request, so a re-run or a crash never repeats work;
- runs the **health gate** before each uncached batch and stops rather than piling onto a sick service;
- **retries** a failed batch with exponential backoff, and pauses between batches.

In [7]:
CACHE_DIR = Path(os.environ.get("BIOMAPPER_TUTORIAL_CACHE", Path.home() / ".cache" / "biomapper_api_tutorial"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REQUEST_LOG = []  # (request_id, n_entities, seconds, from_cache) for the provenance record


class ServiceUnhealthy(RuntimeError):
    pass


def _cache_path(body):
    key = hashlib.sha256(json.dumps({"url": BASE_URL, "body": body}, sort_keys=True).encode()).hexdigest()
    return CACHE_DIR / f"{key}.json"


def map_batch(entities, batch_size=10, pause_s=1.0, retries=3, timeout=600):
    results = []
    for start in range(0, len(entities), batch_size):
        body = {"entities": entities[start : start + batch_size]}
        path = _cache_path(body)
        if path.exists():
            payload = json.loads(path.read_text())
            REQUEST_LOG.append((payload["metadata"]["request_id"], len(body["entities"]), 0.0, True))
            results.extend(payload["results"])
            continue
        ok, detail = api_health()
        if not ok:
            raise ServiceUnhealthy(f"health gate failed before batch at {start}: {detail}")
        for attempt in range(retries + 1):
            try:
                t0 = time.time()
                r = session.post(f"{BASE_URL}/map/batch", json=body, timeout=timeout)
                if r.status_code >= 500:
                    raise requests.HTTPError(f"{r.status_code} from server")
                r.raise_for_status()  # a 4xx is a bad request: fail now, do not retry
                break
            except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as exc:
                if attempt == retries or (
                    getattr(exc, "response", None) is not None and exc.response.status_code < 500
                ):
                    raise
                time.sleep(2 ** (attempt + 1))
        payload = r.json()
        path.write_text(json.dumps(payload))
        REQUEST_LOG.append((payload["metadata"]["request_id"], len(body["entities"]), time.time() - t0, False))
        results.extend(payload["results"])
        time.sleep(pause_s)
    return results


def summarize(results, target=None):
    rows = []
    for x in results:
        eq = x.get("kg_equivalent_ids") or {}
        row = {
            "name": x["name"],
            "chosen_kg_id": x["chosen_kg_id"],
            "certificate": (x.get("resolution_certificate") or {}).get("state"),
            "error": x.get("error"),
        }
        if target:
            row[f"{target} ids"] = ", ".join(eq.get(target, [])[:3])
        rows.append(row)
    return pd.DataFrame(rows)

## 5. Resolving across entity types

One batch, four entity types. Each entity carries its own `entity_type`. Supplying an identifier you already have (the UniProt accession for IL-6 below) goes in `identifiers`.

In [8]:
mixed = [
    {"name": "carnitine", "entity_type": "metabolite"},
    {"name": "hippurate", "entity_type": "metabolite"},
    {"name": "cholesterol", "entity_type": "metabolite"},
    {"name": "Interleukin-6", "entity_type": "protein"},
    {"name": "Interleukin-6", "entity_type": "protein", "identifiers": {"UniProtKB": "P05231"}},
    {"name": "Insulin", "entity_type": "protein"},
    {"name": "Insulin", "entity_type": "protein", "identifiers": {"UniProtKB": "P01308"}},
    {"name": "TP53", "entity_type": "gene"},
    {"name": "APOE", "entity_type": "gene"},
    {"name": "hemoglobin A1c", "entity_type": "lab"},
    {"name": "LDL cholesterol", "entity_type": "lab"},
]
t0 = time.time()
mixed_results = map_batch(mixed)
print(f"{len(mixed_results)} entities in {time.time() - t0:.1f} s")
df = summarize(mixed_results)
df.insert(1, "entity_type", [e["entity_type"] for e in mixed])
EXPECTED_ACCESSION = {"Interleukin-6": "P05231", "Insulin": "P01308"}  # the reviewed UniProt entry for each
df["expected UniProt in xrefs"] = [
    (
        (EXPECTED_ACCESSION[e["name"]] in (x.get("kg_equivalent_ids") or {}).get("UniProtKB", []))
        if e["entity_type"] == "protein"
        else None
    )
    for x, e in zip(mixed_results, mixed)
]
df

11 entities in 2.2 s


,name,entity_type,chosen_kg_id,certificate,error,expected UniProt in xrefs
0,carnitine,metabolite,RM:0008606,corroborated,None,None
1,hippurate,metabolite,RM:0135980,corroborated,None,None
2,cholesterol,metabolite,RM:0007253,unavailable,None,None
3,Interleukin-6,protein,HGNC:6018,not_applicable,None,True
4,Interleukin-6,protein,HGNC:6018,not_applicable,None,True
5,Insulin,protein,LOINC:LP14676-8,not_applicable,None,False
6,Insulin,protein,HGNC:6081,not_applicable,None,True
7,TP53,gene,HGNC:11998,not_applicable,None,None
8,APOE,gene,HGNC:613,not_applicable,None,None
9,hemoglobin A1c,lab,UMLS:C1261236,not_applicable,None,None


Things to notice:

- Only the small molecules carry a certificate state other than `not_applicable`.
- Proteins resolve to the **HGNC gene node** that the graph uses for the gene and its product; the UniProt accessions are in that node's `kg_equivalent_ids`.
- **`Insulin` given by name alone does not reach the protein.** It resolves to a LOINC analyte concept, because "insulin" is also a lab test name. Supplying the accession (`P01308`) fixes it, and so would the gene symbol `INS` or requesting `UniProtKB` as the target namespace (section 6). A bare protein name that doubles as a lab analyte or a drug is ambiguous; send an identifier when you have one, and check `chosen_kg_id`'s prefix.
- Clinical labs resolve to LOINC or UMLS concepts. Check these by eye: lab names are short and ambiguous, and the certificate does not cover them.

## 6. Requesting a target namespace

`options.vocab` names the namespace you want identifiers in, for example `ENSEMBL`, `NCBIGene` or `UniProtKB` for genes. It is not only a filter on the output: **it steers which node is chosen.** For an ordinary protein-coding gene all three targets agree.

In [9]:
TARGETS = ["ENSEMBL", "NCBIGene", "UniProtKB"]


def resolve_per_target(names, entity_type="gene"):
    rows = []
    for target in TARGETS:
        out = map_batch([{"name": n, "entity_type": entity_type, "options": {"vocab": target}} for n in names])
        for x in out:
            eq = x.get("kg_equivalent_ids") or {}
            ids = [i for i in eq.get(target, []) if target != "ENSEMBL" or i.startswith("ENSG")]
            rows.append(
                {
                    "name": x["name"],
                    "target": target,
                    "chosen_kg_id": x["chosen_kg_id"],
                    "target ids returned": ", ".join(ids[:3]),
                }
            )
    return pd.DataFrame(rows)


resolve_per_target(["TP53"])

,name,target,chosen_kg_id,target ids returned
0,TP53,ENSEMBL,HGNC:11998,ENSG00000141510
1,TP53,NCBIGene,HGNC:11998,7157
2,TP53,UniProtKB,HGNC:11998,"A0A087WT22, A0A087WXZ1, A0A087X1Q1"


### A known limitation for genes: forced hits

When the gene you asked for has **no identifier in the target namespace**, the engine currently does not abstain. It moves to a different gene that does have one, and returns that gene's identifier. We call this a forced hit.

`TRK-CTT2-5` is a tRNA gene (NCBI Gene 100189282). It has no Ensembl gene ID and no UniProt entry. `ACTR2P1` is a pseudogene of `ACTR2`. Watch what each target does:

In [10]:
forced = resolve_per_target(["TRK-CTT2-5", "ACTR2P1"])
forced

,name,target,chosen_kg_id,target ids returned
0,TRK-CTT2-5,ENSEMBL,HGNC:6541,ENSG00000111716
1,ACTR2P1,ENSEMBL,HGNC:169,ENSG00000138071
2,TRK-CTT2-5,NCBIGene,HGNC:34848,100189282
3,ACTR2P1,NCBIGene,HGNC:56978,100421614
4,TRK-CTT2-5,UniProtKB,HGNC:6541,"A0A5F9ZHM4, P07195, Q5U077"
5,ACTR2P1,UniProtKB,HGNC:169,P61160


With `NCBIGene` as the target, both resolve to their own nodes (`HGNC:34848` for the tRNA, `HGNC:56978` for the pseudogene). With `ENSEMBL` or `UniProtKB` as the target, both land on a **different, protein-coding gene**, whose identifiers come back looking like a normal answer. The pseudogene lands on its parent `ACTR2` (`HGNC:169`); the tRNA lands on an unrelated gene. Nothing in the response marks these rows as suspect.

How often this happens, on a 1,500-symbol HGNC benchmark scored per target namespace (each namespace scored from the run that targeted it):

| Target namespace | Agreement with HGNC's cross-reference | Symbols with no ID in that namespace that still got one | ...of which the wrong gene, where checkable |
|---|---|---|---|
| Ensembl | 95.0% (1,343/1,413) | 87 | 67 of 84 |
| NCBI Gene | 98.5% (1,453/1,475) | 25 | 19 of 22 |
| UniProtKB | 92.6% (617/666) | 834 | 829 of 831 |

These are **coverage** figures, not independent accuracy: the NCBI Gene source is ingested into the graph, so gold and answer share a source. Forced hits sit outside the agreement denominators, which is why a high agreement rate can coexist with them. The numbers are from benchmark run `suite_20260927T180639Z` against kg 2.3.0, re-scored by `biomapper` commit `3761dc87`. This is reported as a known limitation; an engine-side fix that abstains instead is planned.

Until then, a cheap client-side guard: **resolve under more than one target and flag disagreement.** If the chosen node changes with the target, at least one answer is a forced hit.

In [11]:
def flag_target_disagreement(per_target):
    nodes = per_target.groupby("name")["chosen_kg_id"].agg(lambda s: sorted(set(s)))
    return pd.DataFrame(
        {
            "chosen nodes across targets": nodes,
            "flag": nodes.map(lambda n: "REVIEW: target-dependent" if len(n) > 1 else "consistent"),
        }
    )


flag_target_disagreement(pd.concat([resolve_per_target(["TP53"]), forced]))

,chosen nodes across targets,flag
name,,
ACTR2P1,"[HGNC:169, HGNC:56978]",REVIEW: target-dependent
TP53,[HGNC:11998],consistent
TRK-CTT2-5,"[HGNC:34848, HGNC:6541]",REVIEW: target-dependent


The guard is conservative in one direction only: a disagreement is always worth a look, but agreement does not prove the answer right.

## 7. Record provenance for your run

Save this next to your results. It holds the graph pins, the API you used, and every request id, which is what someone needs to reproduce or audit the mapping. The API key, if any, is deliberately not written.

In [12]:
record = {
    "executed_at": datetime.now(timezone.utc).isoformat(),
    "api_url": BASE_URL,
    "api_health": api_health()[1],
    "kestrel_url": KESTREL_URL,
    "kg_pins": {k: v for k, v in PINS.items() if k != "sources"},
    "kg_sources": PINS["sources"],
    "api_key_used": bool(API_KEY),
    "requests": [
        {"request_id": rid, "n_entities": n, "seconds": round(s, 2), "from_cache": c} for rid, n, s, c in REQUEST_LOG
    ],
}
record_path = CACHE_DIR / "provenance_record.json"
record_path.write_text(json.dumps(record, indent=1))
live = [r for r in record["requests"] if not r["from_cache"]]
print(f"{len(record['requests'])} batch requests ({len(live)} live, {len(record['requests']) - len(live)} from cache)")
if live:
    per_entity = sum(r["seconds"] for r in live) / sum(r["n_entities"] for r in live)
    print(f"live wall time this run: {per_entity:.2f} s per entity")
    print(
        "(the server caches recent lookups for about an hour, so a repeat run is fast; plan for ~4.4 s per entity cold)"
    )
print("saved:", str(record_path).replace(str(Path.home()), "~"))

11 batch requests (8 live, 3 from cache)
live wall time this run: 0.03 s per entity
(the server caches recent lookups for about an hour, so a repeat run is fast; plan for ~4.4 s per entity cold)
saved: ~/.cache/biomapper_api_tutorial/provenance_record.json


## Where next

- **Whole files:** `POST /map/dataset` maps an uploaded TSV/CSV and returns a server-side output path; `POST /map/dataset/stream` streams per-entity results as NDJSON, which is the better choice from a client.
- **Auditing a choice:** `options.kestrel_top_n` returns the raw Kestrel rows behind a result without changing `chosen_kg_id` (see the README).
- **Interactive docs:** the OpenAPI/Swagger UI for the hosted service is at `https://biomapper.expertintheloop.io/api/v1/docs`.
- **Python client:** the [`biomapper` package](https://github.com/trentleslie/biomapper) and its [tutorial](https://github.com/trentleslie/biomapper/blob/main/notebooks/biomapper_tutorial.ipynb).